# Algorithmic Practice Lab and Machine Learning Applications

**Session:** 2026-11-02 · Recursion and Two Pointer  
**Module:** 01 · Python Fundamentals & Algorithmic Thinking

This lab bridges theoretical recursion and two-pointer algorithms to core data processing and machine learning workflows:
1. **Recursive Decision Tree Building:** Recursive partitioning of feature space using Gini impurity.
2. **Two-Pointer Streaming Merge:** Merging sorted feature timestamps and inverted index posting lists in $O(n + m)$.
3. **Comprehensive Unit Testing Suite:** Validating edge cases and asymptotic properties.


## 1. Machine Learning Case Study: Recursive Decision Tree Construction

In machine learning, Decision Tree classifiers (CART) are constructed via **recursive divide-and-conquer**:
1. Base cases: If all samples in current node belong to the same class (purity), or if maximum depth is reached, create a leaf node.
2. Recursive step: Find feature and threshold that minimize weighted Gini impurity, split dataset into left and right partitions, and recursively build `node.left` and `node.right`.


In [ ]:
from dataclasses import dataclass
from typing import Optional


@dataclass
class TreeNode:
    feature: int | None = None
    threshold: float | None = None
    left: Optional["TreeNode"] = None
    right: Optional["TreeNode"] = None
    value: int | None = None

    @property
    def is_leaf(self) -> bool:
        return self.value is not None


def gini_impurity(labels: list[int]) -> float:
    if not labels:
        return 0.0
    total = len(labels)
    counts: dict[int, int] = {}
    for y_val in labels:
        counts[y_val] = counts.get(y_val, 0) + 1
    return 1.0 - sum((cnt / total) ** 2 for cnt in counts.values())


def build_decision_tree(
    samples: list[list[float]],
    labels: list[int],
    depth: int = 0,
    max_depth: int = 2,
) -> TreeNode:
    """Recursively train a binary decision tree classifier."""
    # Base Case 1: Pure node or max depth reached
    if len(set(labels)) <= 1 or depth >= max_depth:
        majority_class = max(set(labels), key=labels.count) if labels else 0
        return TreeNode(value=majority_class)

    num_features = len(samples[0])
    best_gini = float("inf")
    best_split = None

    # Greedy search over features and split thresholds
    for feat in range(num_features):
        values = sorted({row[feat] for row in samples})
        for i in range(len(values) - 1):
            threshold = (values[i] + values[i + 1]) / 2.0
            left_labels = [labels[k] for k in range(len(labels)) if samples[k][feat] <= threshold]
            right_labels = [labels[k] for k in range(len(labels)) if samples[k][feat] > threshold]
            if not left_labels or not right_labels:
                continue
            weighted_gini = (
                len(left_labels) * gini_impurity(left_labels)
                + len(right_labels) * gini_impurity(right_labels)
            ) / len(labels)
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_split = (feat, threshold)

    if best_split is None:
        return TreeNode(value=max(set(labels), key=labels.count))

    feat, thresh = best_split
    left_samples = [row for row in samples if row[feat] <= thresh]
    left_labels = [labels[k] for k in range(len(labels)) if samples[k][feat] <= thresh]
    right_samples = [row for row in samples if row[feat] > thresh]
    right_labels = [labels[k] for k in range(len(labels)) if samples[k][feat] > thresh]

    # Recursive descent into subtrees
    left_child = build_decision_tree(left_samples, left_labels, depth + 1, max_depth)
    right_child = build_decision_tree(right_samples, right_labels, depth + 1, max_depth)

    return TreeNode(feature=feat, threshold=thresh, left=left_child, right=right_child)


def predict_tree(node: TreeNode, sample: list[float]) -> int:
    """Recursively classify a sample through the decision tree."""
    if node.is_leaf:
        assert node.value is not None
        return node.value
    assert node.feature is not None and node.threshold is not None
    if sample[node.feature] <= node.threshold:
        assert node.left is not None
        return predict_tree(node.left, sample)
    else:
        assert node.right is not None
        return predict_tree(node.right, sample)


# Train on synthetic separable dataset
samples_train = [[1.0, 1.0], [1.5, 2.0], [5.0, 5.0], [6.0, 5.5]]
labels_train = [0, 0, 1, 1]
tree = build_decision_tree(samples_train, labels_train, max_depth=2)
print("Tree root split on feature:", tree.feature, "threshold:", tree.threshold)
assert predict_tree(tree, [1.2, 1.5]) == 0
assert predict_tree(tree, [5.5, 5.2]) == 1

## 2. Two-Pointer Merging: Sorted Stream Intersection and Merging

In information retrieval, search engines, and RAG pipelines, inverted index postings (lists of document IDs containing terms) are sorted. Finding documents matching term A AND term B is computed using two pointers in $O(n + m)$ without re-sorting.


In [ ]:
from collections.abc import Sequence


def intersect_sorted_postings(postings_a: Sequence[int], postings_b: Sequence[int]) -> list[int]:
    """Find common document IDs across two sorted posting lists.

    Time Complexity: O(|A| + |B|)
    Auxiliary Space: O(min(|A|, |B|))
    """
    i, j = 0, 0
    common = []
    while i < len(postings_a) and j < len(postings_b):
        if postings_a[i] == postings_b[j]:
            common.append(postings_a[i])
            i += 1
            j += 1
        elif postings_a[i] < postings_b[j]:
            i += 1
        else:
            j += 1
    return common


def merge_sorted_streams(stream_a: Sequence[int], stream_b: Sequence[int]) -> list[int]:
    """Merge two sorted sequences into a single sorted sequence in O(n + m)."""
    i, j = 0, 0
    merged = []
    while i < len(stream_a) and j < len(stream_b):
        if stream_a[i] <= stream_b[j]:
            merged.append(stream_a[i])
            i += 1
        else:
            merged.append(stream_b[j])
            j += 1
    merged.extend(stream_a[i:])
    merged.extend(stream_b[j:])
    return merged


postings_doc1 = [101, 105, 120, 230, 450]
postings_doc2 = [105, 110, 230, 300, 450]
common_docs = intersect_sorted_postings(postings_doc1, postings_doc2)
print("Common documents matching both terms:", common_docs)
assert common_docs == [105, 230, 450]

merged_docs = merge_sorted_streams(postings_doc1, postings_doc2)
assert merged_docs == sorted(postings_doc1 + postings_doc2)

## 3. Comprehensive Verification Suite


In [ ]:
import unittest


class RecursionAndTwoPointersSuite(unittest.TestCase):
    def test_postings_intersection(self):
        self.assertEqual(intersect_sorted_postings([1, 2, 3], [2, 3, 4]), [2, 3])

    def test_stream_merging(self):
        self.assertEqual(
            merge_sorted_streams([1, 4, 7], [2, 3, 5, 8]),
            [1, 2, 3, 4, 5, 7, 8],
        )


suite = unittest.TestLoader().loadTestsFromTestCase(RecursionAndTwoPointersSuite)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful()